# SOC Security Framework — Evaluation Tests T00–T05

Runs the exact same tests as `run_tests.py` via the persistent framework server.
Each cell calls a `/run_test/TXX` endpoint that executes the test server-side.

In [ ]:
import os, requests, json
from IPython.display import display, HTML

BASE = os.getenv('FRAMEWORK_SERVER_URL', 'http://framework-server:8090')
N    = 20

def run_test(tid, n=N):
    r = requests.post(f'{BASE}/run_test/{tid}', json={'n': n}, timeout=1000)
    return r.json()

def show(r):
    if 'error' in r:
        print(f'ERROR: {r["error"]}')
        return
    ok = r['attacks_passed'] == 0 and r['false_blocks'] == 0
    color = 'green' if ok else 'red'
    pct = f"{r['attacks_blocked']/r['attacks_total']*100:.0f}%" if r['attacks_total'] else 'N/A'
    ovh = f"{r['overhead_ms']:+.1f}ms" if r.get('overhead_ms') else 'N/A'
    html = f"""
    <table style='border-collapse:collapse;font-family:monospace;font-size:13px;margin:8px 0'>
    <tr><th colspan=2 style='background:#1a1a2e;color:#e0e0e0;padding:8px 12px;text-align:left'>
        {r['test_id']}: {r['test_name']} &nbsp;·&nbsp; <span style='font-weight:normal'>{r['layer']}</span></th></tr>
    <tr><td style='padding:3px 12px;color:#888'>Attack vectors</td>
        <td style='padding:3px 8px'>{', '.join(r['attack_vectors'])}</td></tr>
    <tr><td style='padding:3px 12px;color:#888'>Trials</td>
        <td style='padding:3px 8px'>{r['n_trials']}</td></tr>
    <tr><td style='padding:3px 12px;color:#888'>Attacks total</td>
        <td style='padding:3px 8px'><b>{r['attacks_total']}</b></td></tr>
    <tr><td style='padding:3px 12px;color:#888'>Blocked</td>
        <td style='padding:3px 8px;color:green'><b>{r['attacks_blocked']} ({pct})</b></td></tr>
    <tr><td style='padding:3px 12px;color:#888'>Passed (miss)</td>
        <td style='padding:3px 8px;color:{color}'><b>{r['attacks_passed']}</b> {'✓' if r['attacks_passed']==0 else '✗'}</td></tr>
    <tr><td style='padding:3px 12px;color:#888'>False blocks</td>
        <td style='padding:3px 8px;color:{color}'><b>{r['false_blocks']}</b> {'✓' if r['false_blocks']==0 else '✗'}</td></tr>
    <tr><td style='padding:3px 12px;color:#888'>Legit passed</td>
        <td style='padding:3px 8px'>{r['legit_passed']}</td></tr>
    <tr><td style='padding:3px 12px;color:#888'>Overhead</td>
        <td style='padding:3px 8px'>{ovh}</td></tr>
    <tr><td style='padding:3px 12px;color:#888'>Notes</td>
        <td style='padding:3px 8px;color:#666'>{r.get('notes','')}</td></tr>
    </table>"""
    display(HTML(html))

status = requests.get(f'{BASE}/status').json()
print(f'Framework: {status["status"]} | Session: {status["session_id"][:16]} | Trials: {N}')

## T00 — Tool Registration Validator

In [ ]:
show(run_test('T00'))

## T01 — Layer 1: Access Control

In [ ]:
show(run_test('T01'))

## T02 — Layer 2: Rate Limiter

In [ ]:
show(run_test('T02'))

## T03 — Layer 3: Input Validator

In [ ]:
show(run_test('T03'))

## T04 — Layer 4: Output Validator

In [ ]:
show(run_test('T04'))

## T05 — Session and Response Controls

In [ ]:
show(run_test('T05'))